# Using the Solver #

Today we will talk about dictionaries and we will talk about how to use the PULP solver to solve LPs.

## Dictionaries ##
They store a collection of key/value pairs.These are useful for storing named object and for "looking up" values.

In [1]:
#This is an example of a dictionary
mydictionary = {"Patrick":"Star","Terry":"Tao","John":"Conway","Emmy":"Noether"}
#This returns the value associated with the key
print(mydictionary["Terry"])

#This returns the keys from the dictionary.
#You can use these keys to get objects from the dictionary
keys = mydictionary.keys()
print(keys)
#This allows us to get the dictionary values. We will not get their corresponding keys though.
vals = mydictionary.values()
print(vals)

#Add to the dictionary
mydictionary["Bob"] = "Marley"
print(mydictionary)

#Update the dictionary
mydictionary["Patrick"] = "Swayze"
print(mydictionary)

Tao
dict_keys(['Patrick', 'Terry', 'John', 'Emmy'])
dict_values(['Star', 'Tao', 'Conway', 'Noether'])
{'Patrick': 'Star', 'Terry': 'Tao', 'John': 'Conway', 'Emmy': 'Noether', 'Bob': 'Marley'}
{'Patrick': 'Swayze', 'Terry': 'Tao', 'John': 'Conway', 'Emmy': 'Noether', 'Bob': 'Marley'}


In [2]:
#Dictionaries can be used in place of a list (but usually lists are better).
#You might opt to use a dictionary when implementing simplex method if you want to keep track of
#the active/inactive constraints.
indexing = {1:67,2:43,3:990}

print(indexing[2])

#Dictionaries can contain multiple object types
mixeddictionary = {1:6, "a":4, 5:"abc"}
mixeddictionary["a"]

#Dictionaries can contain lists and dictionaries!
alphabet = {"a":["abacus","alfalfa","atlantic","average"],"b":["barley","bear","beast","beauty"]}
print(alphabet["a"][2])

#We have a nested dictionary here
nesteddictionary = {"birds":{"raptors":["eagle","hawk","falcon"],"other":['stark','chicken','turkeys']},"dictionary":"Not a bird"}
print(nesteddictionary["birds"]["raptors"][2])

#We update by using the appropriate number of [] when nesting.
nesteddictionary["birds"]["other"] = "turkey"
nesteddictionary["birds"]["raptors"].append("owl")
print(nesteddictionary)

43
atlantic
falcon
{'birds': {'raptors': ['eagle', 'hawk', 'falcon', 'owl'], 'other': 'turkey'}, 'dictionary': 'Not a bird'}


## Using PULP ##

Pulp is the library we will use to solve the LPs. You can use PULP to create .lp files. These then get fed into a solver. The default solver is free, but if you go to industry, you could feed the .lp files into standard paid solvers like CPLEX or Gurobi.

In [3]:
!pip install pulp


In [4]:
#Here we are importing pulp
import pulp

We are going to work with a problem about optimizing the cost of a catfood, provided that the catfood must meet certain nutritional requirements. The formulation isn't so important in this example, and if you want to see specifics, I suggest going to the original documentation: https://coin-or.github.io/pulp/CaseStudies/a_blending_problem.html

In [5]:
#Create an LP
ourproblem = pulp.LpProblem("CatFoodProblem", pulp.LpMinimize)

#Create the decision variables. We have the lower bound, upper bound, and variable type
#If no lower/upper bound is given, then the variable is assumed to be unbounded on that side
x1 = pulp.LpVariable("CatFood1", lowBound=0, upBound=100, cat=pulp.LpContinuous)
x2 = pulp.LpVariable("CatFood2", lowBound=0, upBound=100, cat=pulp.LpContinuous)

#Create the objective
ourproblem += 10*x1 + 12*x2

#Create the constraints

#Percentage Constraint
ourproblem += x1 + x2 == 100

#Fat Constraint
ourproblem += 0.3*x1 + 0.1*x2 >= 8

#Protein Constraint
ourproblem += 2.0*x1 + 4.0*x2 >= 300

#Write the LP
ourproblem.writeLP("CatFoodProblem.lp")
ourproblem.solve()

#Check if the solution is optimal/infeasible
print(pulp.LpStatus[ourproblem.status])

#Get the variable values
varvalues = {}
for v in ourproblem.variables():
    varvalues[v.name] = v.varValue
    print(v.name, "=", v.varValue)

print(varvalues)

Optimal
CatFood1 = 50.0
CatFood2 = 50.0
{'CatFood1': 50.0, 'CatFood2': 50.0}


## Bigger Example ##

If you store the names of variables and corresponding constraint values in dictionaries, you can automate the creation of constraints. This is particularly useful for larger problems.

In [6]:
#Ingredients and Constraint Values
ingredients = ["CHICKEN","BEEF","MUTTON","RICE","WHEAT","GEL"]

#Costs of each ingredient
costs = {
    "CHICKEN":0.013,
    "BEEF":0.008,
    "MUTTON":0.010,
    "RICE":0.002,
    "WHEAT":0.005,
    "GEL":0.001,

}

#Protein percent for each ingredient
proteinPercent = {
    "CHICKEN": 0.100,
    "BEEF": 0.200,
    "MUTTON": 0.150,
    "RICE": 0.000,
    "WHEAT": 0.040,
    "GEL": 0.000,
}

#Fat percent for each ingredient
fatPercent = {
    "CHICKEN": 0.080,
    "BEEF": 0.100,
    "MUTTON": 0.110,
    "RICE": 0.010,
    "WHEAT": 0.010,
    "GEL": 0.000,
}

#Fiber percent for each ingredient
fiberPercent = {
    "CHICKEN": 0.001,
    "BEEF": 0.005,
    "MUTTON": 0.003,
    "RICE": 0.100,
    "WHEAT": 0.150,
    "GEL": 0.000,
}

#Salt percent for each ingredient
saltPercent = {
    "CHICKEN": 0.002,
    "BEEF": 0.005,
    "MUTTON": 0.007,
    "RICE": 0.002,
    "WHEAT": 0.008,
    "GEL": 0.000,
}

#Create the LP
gelproblem = pulp.LpProblem("GelProblem", pulp.LpMinimize)

#Automate name and creation of similar variables
#These are stored in a dictionary for easy reference
ingredient_vars = pulp.LpVariable.dicts("Ingr",ingredients,lowBound = 0, upBound = 100, cat = pulp.LpContinuous)

#Get Objective. We are summing over the objects in the list.
#Here we are using list comprehension for convenience.
gelproblem += pulp.lpSum([costs[i]*ingredient_vars[i] for i in ingredients]),"Total Cost"

#Add Constraints. This process is similar to the objective.
gelproblem += pulp.lpSum([ingredient_vars[i] for i in ingredients]) == 100, "PercentagesSum"
gelproblem += (
    pulp.lpSum([proteinPercent[i] * ingredient_vars[i] for i in ingredients]) >= 8.0,
    "ProteinRequirement",
)
gelproblem += (
    pulp.lpSum([fatPercent[i] * ingredient_vars[i] for i in ingredients]) >= 6.0,
    "FatRequirement",
)
gelproblem += (
    pulp.lpSum([fiberPercent[i] * ingredient_vars[i] for i in ingredients]) <= 2.0,
    "FiberRequirement",
)
gelproblem += (
    pulp.lpSum([saltPercent[i] * ingredient_vars[i] for i in ingredients]) <= 0.4,
    "SaltRequirement",
)

#Write and solve the LP
gelproblem.writeLP("GelProblem.lp")
gelproblem.solve()

#Check if the solution is optimal/infeasible
print(pulp.LpStatus[ourproblem.status])

#Get the variable values
varvalues = {}
for v in gelproblem.variables():
    varvalues[v.name] = v.varValue
    print(v.name, "=", v.varValue)

Optimal
Ingr_BEEF = 60.0
Ingr_CHICKEN = 0.0
Ingr_GEL = 40.0
Ingr_MUTTON = 0.0
Ingr_RICE = 0.0
Ingr_WHEAT = 0.0
